# E07 — Rede GE mesh original

Este notebook usa os 13 sites do example_1 para planejar a rede GE mesh e apoiar a escolha de pontos para acesso celular público. A análise de qualidade e dependências é visual; a decisão pertence ao engenheiro, inclusive quando o ponto escolhido é uma folha importante.

Toda a lógica de planejamento, análise e persistência fica no SDK e no serviço. As células apenas chamam o SDK. As posições e alturas foram preservadas; os parâmetros RF são ilustrativos.

Requer SDK 0.1.38+ e o serviço atualizado. Execute com o kernel na pasta notebooks do workspace.

In [ ]:
from cisei_planning_sdk import PlanningClient, PlanningProject

client = PlanningClient("http://network-service:8080", workspace="..", timeout=1900.0)
project = PlanningProject("../projects/e07_ge_mesh_original")
scenario = client.open_project(project, planner="graph")


## 1. Inspecionar os enlaces candidatos

O grafo permite selecionar um site ou enlace, consultar seus detalhes e navegar pela rede. Os candidatos ainda não são os enlaces que serão instalados.

In [ ]:
scenario.build_candidate_edges();
scenario.show_network(kind="candidate");


## 2. Calcular métricas e planejar a rede original

O serviço extrai as features necessárias; o SDK preserva o cache do projeto. A solução seleciona os enlaces da rede GE. O rank acumulado e o limiar de qualidade são distintos da existência de um caminho.

In [ ]:
scenario.compute_metrics(geo_base_url="http://features:8080", batch_size=10);
scenario.solve();
scenario.save_result(rank_threshold=6.0, primary_tech="ge_mesh");
project.show_network();


## 3. Explorar as dependências no grafo

Selecione um site no grafo ou na busca. A opção **Descendant subnetwork** destaca os enlaces da sub-rede dependente. O painel informa filhos diretos, dispositivos, sites e interfaces descendentes. Desmarque a opção para consultar apenas os enlaces vizinhos.

A qualidade permanece indicada no grafo: magenta identifica custo acima do limiar e vermelho identifica ausência de atendimento. A quantidade de descendentes ajuda a avaliar o alcance da proteção, mas não determina automaticamente quem deve receber celular.

In [ ]:
project.show_dependencies();


## 4. Registrar a escolha manual

Neste exemplo, o engenheiro escolhe barigui após inspecionar a sub-rede. Substitua o argumento por outro site se a prioridade de operação for diferente. Uma folha também pode ser selecionada.

O SDK registra a escolha e a identidade do resultado que a fundamentou. Essa chamada não troca perfis, não muda enlaces e não executa novamente o planejamento. A configuração de um dispositivo com interfaces GE e celular será tratada no exemplo seguinte.

In [ ]:
project.save_redundancy_selection("barigui");
